# Tasks 4.4, 4.5, 4.6, and 4.10

## Task 4.4
**4.4 Predict magnitude of antibody response - all 3 vaccine strains (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Geo mean / Metric: Spearman correlation
* Full description: Geo mean of HAI across the 3 vaccine strains (4.1, 4.2, 4.3) at Day 28

> **Note — missing strain:** `H3N2 A/Massachusetts/18/2022` has no training measurements (challenge-only strain). This task effectively averages over the 2 vaccine strains that are present in `hai_cleaned.csv` (H1N1 A/Victoria/4897/2022 and Vic B/Austria/1359417/2021), not all 3.

## Task 4.5
**4.5 Predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Geo mean / Metric: Spearman correlation
* Full description: Geomean HAI across all variants at Day 28

## Task 4.6
**4.6 Predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Percentage / Metric: Spearman correlation
* Full description: Percentage of variants with HAI >= 40 at Day 28

## Task 4.10
**4.10 Predict antibody durability - all 3 vaccine strains (D365)**
* Training Data: Demographics + Day 0 + Day 7 innate + **Day 28**
* Assay: HAI / Measure: Geo mean HAI / Metric: Spearman correlation
* Full description: Geo mean of HAI across the 3 vaccine strains at Day 365

> **Note — missing strain:** of the 3 vaccine strains, only `Vic B/Austria/1359417/2021` has a `_d365` column in the training data (H1N1 A/Victoria/4897/2022 stops at d28; H3N2 A/Massachusetts/18/2022 is absent entirely). This task therefore collapses to predicting Vic B D365 — effectively the same target as Task 4.9.

---

## Design notes

**y-values:** log2-transformed. Since Spearman only cares about ranking, no inverse transform is needed for evaluation. Metrics (RMSE, MAE, Spearman) are all in log2 space; CSVs use `np.exp2` to output raw titer scale values (task 4.6 outputs a fraction directly).

**Geo mean on log2 data:** HAI values in the parquet are already log2-transformed, so the arithmetic mean across strain columns equals log2(geometric mean) on the raw titer scale.

**Spearman correlation:** ranks predictions and truth; rewards monotonic agreement regardless of scale. Robust to outliers. Score: 1.0 = perfect, 0.0 = no signal, -1.0 = reversed.

**5-fold cross-validation:** each participant's prediction is made by a model that never saw them during training.

In [1]:
VACCINE_STRAINS = [
    'H1N1 A/Victoria/4897/2022',
    'H3N2 A/Massachusetts/18/2022',
    'Vic B/Austria/1359417/2021',
]
HAI_THRESHOLD = 40  # task 4.6: count variants with raw HAI >= 40
AUTO_ML_MAX_RUNTIME_SECONDS = 1200

In [2]:
PARQUET_PATH = '../merged_data/combined.parquet'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [3]:
import io
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=UserWarning, module="h2o")
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
  Starting server from C:\Users\User\anaconda3\envs\cmi-flu-prediction-challenge-capstone\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\User\AppData\Local\Temp\tmp1jb0r9iv
  JVM stdout: C:\Users\User\AppData\Local\Temp\tmp1jb0r9iv\h2o_User_started_from_python.out
  JVM stderr: C:\Users\User\AppData\Local\Temp\tmp1jb0r9iv\h2o_User_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,America/Los_Angeles
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 16 days
H2O_cluster_name:,H2O_from_python_User_7vns7s
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,15.41 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"


In [4]:
data = h2o.import_file(PARQUET_PATH)
print(f'Training data shape: {data.shape}')

challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
print(f'Challenge shape: {challenge_data.shape}')
challenge_hf = h2o.H2OFrame(challenge_data)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Training data shape: (3757, 110063)
Challenge shape: (40, 23)
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [5]:
# Extract HAI columns from H2O frame to pandas for aggregate target computation
hai_d28_cols = [c for c in data.columns if c.startswith('HAI_') and c.endswith('_d28')]
hai_d365_cols = [c for c in data.columns if c.startswith('HAI_') and c.endswith('_d365')]
vaccine_d28_cols = [c for c in hai_d28_cols if any(s in c for s in VACCINE_STRAINS)]
vaccine_d365_cols = [c for c in hai_d365_cols if any(s in c for s in VACCINE_STRAINS)]

pd_hai_d28 = data[hai_d28_cols].as_data_frame()
pd_hai_d365 = data[hai_d365_cols].as_data_frame()

# Task 4.4: mean of log2 HAI for vaccine strains at D28 == log2(geomean) on raw scale
target_4_4 = pd_hai_d28[vaccine_d28_cols].mean(axis=1, skipna=True)
data['TARGET_4_4'] = h2o.H2OFrame(target_4_4.to_frame(name='TARGET_4_4'))

# Task 4.5: mean of log2 HAI across all variants at D28
target_4_5 = pd_hai_d28.mean(axis=1, skipna=True)
data['TARGET_4_5'] = h2o.H2OFrame(target_4_5.to_frame(name='TARGET_4_5'))

# Task 4.6: fraction of variants with raw HAI >= 40 at D28
threshold_log2 = np.log2(HAI_THRESHOLD)
above = (pd_hai_d28 >= threshold_log2).sum(axis=1)
measured = pd_hai_d28.notna().sum(axis=1)
target_4_6 = (above / measured).where(measured > 0)
data['TARGET_4_6'] = h2o.H2OFrame(target_4_6.to_frame(name='TARGET_4_6'))

# Task 4.10: mean of log2 HAI for vaccine strains at D365
target_4_10 = pd_hai_d365[vaccine_d365_cols].mean(axis=1, skipna=True)
data['TARGET_4_10'] = h2o.H2OFrame(target_4_10.to_frame(name='TARGET_4_10'))

TARGET_COLS = ['TARGET_4_4', 'TARGET_4_5', 'TARGET_4_6', 'TARGET_4_10']
print('Targets computed and added to H2O frame.')
print(f'Task 4.4 training samples: {target_4_4.notna().sum()}')
print(f'Task 4.5 training samples: {target_4_5.notna().sum()}')
print(f'Task 4.6 training samples: {target_4_6.notna().sum()}')
print(f'Task 4.10 training samples: {target_4_10.notna().sum()}')

C:\Users\User\anaconda3\envs\cmi-flu-prediction-challenge-capstone\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
C:\Users\User\anaconda3\envs\cmi-flu-prediction-challenge-capstone\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Targets computed and added to H2O frame.
Task 4.4 training samples: 920
Task 4.5 training samples: 3627
Task 4.6 training samples: 3627
Task 4.10 training samples: 492


---

## Task 4.4 — Geo mean HAI across the 3 vaccine strains (D28)

In [6]:
# Features: d0 + d7 — exclude all d28/d365 targets, computed target cols, and participant_id
x_4_4 = [c for c in data.columns
         if not c.endswith('_d28') and not c.endswith('_d365')
         and c != 'participant_id'
         and c not in TARGET_COLS]
y_4_4 = 'TARGET_4_4'

train_4_4 = data[data[y_4_4].isna() == 0]
print(f'Training samples: {train_4_4.nrows}  |  Features: {len(x_4_4)}')

aml_4_4 = H2OAutoML(max_models=10, seed=1, nfolds=5,
                    keep_cross_validation_predictions=True,
                    max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml_4_4.train(x=x_4_4, y=y_4_4, training_frame=train_4_4)
print('Training complete.')

Training samples: 3757  |  Features: 109935
Training complete.


In [7]:
lb = aml_4_4.leaderboard
print(lb.head(rows=lb.nrows))

H2OFrame is empty.


In [8]:
cv_preds = aml_4_4.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train_4_4[y_4_4].as_data_frame()[y_4_4]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.4 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

AttributeError: 'NoneType' object has no attribute 'cross_validation_holdout_predictions'

In [ ]:
print(f'Leader model: {aml_4_4.leader.model_id}')
varimp_4_4 = aml_4_4.leader.varimp(use_pandas=True)
display(varimp_4_4.head(20))
aml_4_4.leader.varimp_plot(num_of_features=20)

In [ ]:
y_pred_challenge = aml_4_4.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.4': np.exp2(y_pred_challenge),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_4.csv', index=False)
results

---

## Task 4.5 — Geo mean HAI across all variants (D28)

In [ ]:
# Features: d0 + d7 — same exclusions as task 4.4
x_4_5 = x_4_4
y_4_5 = 'TARGET_4_5'

train_4_5 = data[data[y_4_5].isna() == 0]
print(f'Training samples: {train_4_5.nrows}  |  Features: {len(x_4_5)}')

aml_4_5 = H2OAutoML(max_models=10, seed=1, nfolds=5,
                    keep_cross_validation_predictions=True,
                    max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml_4_5.train(x=x_4_5, y=y_4_5, training_frame=train_4_5)
print('Training complete.')

In [ ]:
lb = aml_4_5.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml_4_5.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train_4_5[y_4_5].as_data_frame()[y_4_5]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.5 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml_4_5.leader.model_id}')
varimp_4_5 = aml_4_5.leader.varimp(use_pandas=True)
display(varimp_4_5.head(20))
aml_4_5.leader.varimp_plot(num_of_features=20)

In [ ]:
y_pred_challenge = aml_4_5.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.5': np.exp2(y_pred_challenge),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_5.csv', index=False)
results

---

## Task 4.6 — Percentage of variants with HAI >= 40 (D28)

In [ ]:
# Features: d0 + d7 — same exclusions as task 4.4
x_4_6 = x_4_4
y_4_6 = 'TARGET_4_6'

train_4_6 = data[data[y_4_6].isna() == 0]
print(f'Training samples: {train_4_6.nrows}  |  Features: {len(x_4_6)}')

aml_4_6 = H2OAutoML(max_models=10, seed=1, nfolds=5,
                    keep_cross_validation_predictions=True,
                    max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml_4_6.train(x=x_4_6, y=y_4_6, training_frame=train_4_6)
print('Training complete.')

In [ ]:
lb = aml_4_6.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml_4_6.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train_4_6[y_4_6].as_data_frame()[y_4_6]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.6 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml_4_6.leader.model_id}')
varimp_4_6 = aml_4_6.leader.varimp(use_pandas=True)
display(varimp_4_6.head(20))
aml_4_6.leader.varimp_plot(num_of_features=20)

In [ ]:
y_pred_challenge = aml_4_6.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.6': y_pred_challenge,  # fraction — no inverse transform
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_6.csv', index=False)
results

---

## Task 4.10 — Geo mean HAI across the 3 vaccine strains (D365)

In [ ]:
# Features: d0 + d7 + d28 allowed for durability task — exclude only d365 targets
x_4_10 = [c for c in data.columns
          if not c.endswith('_d365')
          and c != 'participant_id'
          and c not in TARGET_COLS]
y_4_10 = 'TARGET_4_10'

train_4_10 = data[data[y_4_10].isna() == 0]
print(f'Training samples: {train_4_10.nrows}  |  Features: {len(x_4_10)}')

aml_4_10 = H2OAutoML(max_models=10, seed=1, nfolds=5,
                     keep_cross_validation_predictions=True,
                     max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml_4_10.train(x=x_4_10, y=y_4_10, training_frame=train_4_10)
print('Training complete.')

In [ ]:
lb = aml_4_10.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml_4_10.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train_4_10[y_4_10].as_data_frame()[y_4_10]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.10 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml_4_10.leader.model_id}')
varimp_4_10 = aml_4_10.leader.varimp(use_pandas=True)
display(varimp_4_10.head(20))
aml_4_10.leader.varimp_plot(num_of_features=20)

In [ ]:
y_pred_challenge = aml_4_10.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.10': np.exp2(y_pred_challenge),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_10.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---

## Conclusion

### Task 4.4 — Geo mean HAI across vaccine strains at Day 28
- Aggregate of the 3 vaccine strains (effectively 2 due to missing H3N2 A/Massachusetts/18/2022 training data).
- Target: arithmetic mean of log2 HAI at D28 across available vaccine strains = log2(geometric mean).

### Task 4.5 — Geo mean HAI across all variants at Day 28
- Broader breadth target: average over all measured HAI strains at D28.
- Target: arithmetic mean of log2 HAI at D28 across all variants.

### Task 4.6 — Fraction of variants with HAI >= 40 at Day 28
- Binary threshold breadth metric.
- Target: fraction of measured variants where log2(HAI) >= log2(40) ≈ 5.32.
- Challenge CSV stores predicted fraction directly (no exp2 inverse).

### Task 4.10 — Geo mean HAI across vaccine strains at Day 365 (Durability)
- Day 28 features allowed (durability task). Collapses to Vic B D365 due to missing D365 measurements for the other vaccine strains.
- Target: arithmetic mean of log2 HAI at D365 across available vaccine strains.

### Notes
- Training data: `merged_data/combined.parquet` (full feature set including transcriptomics)
- Challenge predictions written to `automl_submission/task_4_4.csv`, `task_4_5.csv`, `task_4_6.csv`, `task_4_10.csv`
- Runtime was capped at `AUTO_ML_MAX_RUNTIME_SECONDS` — increasing this would allow more models (including stacked ensembles) to train and may improve scores further.